# 05 · Pipeline và chia dữ liệu

Challenge 2 – phân loại bài hát theo `Class` (11 lớp). Notebook này kiểm tra đầu vào đã xử lý, tạo holdout theo nhóm bài hát để tránh một bài xuất hiện ở cả train và validation, đồng thời định nghĩa các fold GroupKFold dùng thống nhất cho các notebook mô hình. Tập test chỉ dùng suy luận cuối cùng.

Đầu vào dự kiến: `DATA_CHALLENGE2/processed/train_fe.csv`, `test_fe.csv`, `feature_cols_fe.csv`. Dữ liệu được coi là chỉ đọc; kết quả, nếu tạo, lưu ở thư mục output riêng.

In [1]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
RUN_START = datetime.now()
# Sửa đường dẫn này nếu bộ dữ liệu nằm ngoài thư mục dự án.
# Dữ liệu tham chiếu tại thư mục Challenge 2 trên Desktop.
ROOT = Path(r"C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2")
DATA_DIR = ROOT / "DATA_CHALLENGE2"
PROC_DIR = DATA_DIR / "processed"
LOG_PATH = ROOT / "logs" / "experiment_log.csv"
from sklearn.model_selection import GroupShuffleSplit

TRAIN_PATH = PROC_DIR / "train_processed.csv"
TEST_PATH = PROC_DIR / "test_processed.csv"
TRAIN_FE_PATH = PROC_DIR / "train_fe.csv"
TEST_FE_PATH = PROC_DIR / "test_fe.csv"
COLS_PATH = PROC_DIR / "feature_cols_fe.csv"
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy {TRAIN_PATH}. Đặt dữ liệu Challenge 2 theo cấu trúc DATA_CHALLENGE2/processed hoặc cập nhật DATA_DIR.")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train_fe = pd.read_csv(TRAIN_FE_PATH)
test_fe = pd.read_csv(TEST_FE_PATH)
feature_cols_fe = pd.read_csv(COLS_PATH)["feature"].dropna().astype(str).tolist()
TARGET = "Class"
GROUP = "group_id"
feature_cols = [c for c in train if c not in ("Id", TARGET, GROUP)]
assert TARGET in train and GROUP in train, "Cần có cột Class và group_id trong train_processed.csv"
assert set(feature_cols_fe).issubset(train_fe.columns), "Danh sách feature_cols_fe.csv không khớp train_fe.csv"
assert set(feature_cols_fe).issubset(test_fe.columns), "Danh sách feature_cols_fe.csv không khớp test_fe.csv"
print("Processed train/test:", train.shape, test.shape)
print("Feature-engineered train/test:", train_fe.shape, test_fe.shape)
print("Số lớp:", train[TARGET].nunique(), "| số nhóm:", train[GROUP].nunique(), "| base features:", len(feature_cols), "| FE features:", len(feature_cols_fe))
display(train[TARGET].value_counts().sort_index().rename("count").to_frame())

Processed train/test: (14396, 35) (3600, 33)
Feature-engineered train/test: (14396, 37) (3600, 35)
Số lớp: 11 | số nhóm: 13176 | base features: 32 | FE features: 34


,count
Class,
0,500
1,1098
2,1018
3,322
4,310
5,1157
6,2069
7,461
8,1483


## 1. Holdout theo nhóm
Dùng GroupShuffleSplit thay cho chia ngẫu nhiên theo dòng để không rò rỉ bài hát trùng giữa train và validation. Tỷ lệ có thể lệch lớp do ràng buộc nhóm; kiểm tra phân phối trước khi đánh giá.

In [2]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
fit_idx, valid_idx = next(splitter.split(train[feature_cols], train[TARGET], groups=train[GROUP]))
X_fit, X_valid = train.iloc[fit_idx][feature_cols], train.iloc[valid_idx][feature_cols]
y_fit, y_valid = train.iloc[fit_idx][TARGET], train.iloc[valid_idx][TARGET]
assert set(train.iloc[fit_idx][GROUP]).isdisjoint(set(train.iloc[valid_idx][GROUP]))
print(f"Fit: {len(fit_idx):,} dòng / {train.iloc[fit_idx][GROUP].nunique():,} nhóm")
print(f"Validation: {len(valid_idx):,} dòng / {train.iloc[valid_idx][GROUP].nunique():,} nhóm")
display(pd.DataFrame({"fit": y_fit.value_counts(normalize=True), "validation": y_valid.value_counts(normalize=True)}).fillna(0).sort_index().round(3))

Fit: 11,516 dòng / 10,540 nhóm
Validation: 2,880 dòng / 2,636 nhóm


,fit,validation
Class,,
0,0.035,0.032
1,0.077,0.074
2,0.070,0.075
3,0.023,0.020
4,0.021,0.022
5,0.081,0.076
6,0.141,0.155
7,0.033,0.030
8,0.101,0.111


## 2. Fold GroupKFold dùng chung

In [3]:
N_SPLITS = 5
cv = GroupKFold(n_splits=N_SPLITS)
fold_rows = []
for fold, (tr, va) in enumerate(cv.split(train[feature_cols], train[TARGET], groups=train[GROUP]), start=1):
    assert set(train.iloc[tr][GROUP]).isdisjoint(set(train.iloc[va][GROUP]))
    fold_rows.append({"fold": fold, "train_rows": len(tr), "valid_rows": len(va),
                      "train_groups": train.iloc[tr][GROUP].nunique(), "valid_groups": train.iloc[va][GROUP].nunique(),
                      "valid_classes": train.iloc[va][TARGET].nunique()})
display(pd.DataFrame(fold_rows))

,fold,train_rows,valid_rows,train_groups,valid_groups,valid_classes
0,1,11516,2880,10540,2636,11
1,2,11517,2879,10541,2635,11
2,3,11517,2879,10541,2635,11
3,4,11517,2879,10541,2635,11
4,5,11517,2879,10541,2635,11


## 3. Ghi log

Notebook hiện tại ghi log kết quả mô hình ở 06 và 07. Chạy tuần tự các notebook sau khi dữ liệu đã được đặt đúng vị trí.

In [4]:
from datetime import datetime
import csv, platform, sys
now=datetime.now()
elapsed_sec=round((now-RUN_START).total_seconds(),1)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
record={"run_id":f"05_Pipeline_Split_{now:%Y%m%d_%H%M%S}","date":now.strftime("%Y-%m-%d"),"time":now.strftime("%H:%M:%S"),
 "runner":"Ngô Thanh Phong","stage":"05_Pipeline_Split","notebook":"05_Pipeline_Split.ipynb",
 "goal":"Kiểm tra GroupKFold theo group_id và xác nhận hai bộ processed/feature-engineered",
 "steps":f"base train={train.shape}, test={test.shape}, n_features={len(feature_cols)}; FE train={train_fe.shape}, test={test_fe.shape}, n_features={len(feature_cols_fe)}; 5 folds; mỗi fold có 11 lớp",
 "duration_sec":elapsed_sec,"python":sys.version.split()[0],"platform":platform.platform(),
 "notes":"Fold assignment is group-disjoint; chỉ dùng train để đánh giá; test chưa đánh giá"}
fields=["run_id","date","time","runner","stage","notebook","goal","steps","duration_sec","python","platform","notes"]
exists=LOG_PATH.exists() and LOG_PATH.stat().st_size>0
with LOG_PATH.open("a",newline="",encoding="utf-8-sig") as f:
 writer=csv.DictWriter(f,fieldnames=fields)
 if not exists: writer.writeheader()
 writer.writerow(record)

md_path=LOG_PATH.parent / f"{record['stage']}_{now:%Y%m%d_%H%M%S}_Tô-Doanh-Quang-Khu.md"
suffix=2
while md_path.exists():
    md_path=LOG_PATH.parent / f"{record['stage']}_{now:%Y%m%d_%H%M%S}_Tô-Doanh-Quang-Khu_{suffix}.md"
    suffix+=1
start_text=RUN_START.strftime("%H:%M:%S")
end_text=now.strftime("%H:%M:%S")
steps_md="\n".join(f"- {part.strip()}" for part in record["steps"].split(";"))
md_text=f"""# Nhật ký thí nghiệm – {record['stage']}

| Mục | Giá trị |
|---|---|
| Người chạy | {record['runner']} |
| Ngày chạy | {now.strftime('%d/%m/%Y')} |
| Giờ bắt đầu – kết thúc | {start_text} – {end_text} ({elapsed_sec} giây) |
| Notebook | `{record['notebook']}` |
| Mục tiêu | {record['goal']} |
| Môi trường | Python {sys.version.split()[0]} · {platform.platform()} |

## Các bước & kết quả

{steps_md}

## Nhận xét

{record['notes']}
"""
md_path.write_text(md_text,encoding="utf-8")
record["notes"] += f"; markdown_log={md_path.name}"

print(f"Đã ghi CSV: {LOG_PATH} | Markdown: {md_path}")

Đã ghi CSV: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2\logs\experiment_log.csv | Markdown: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2\logs\05_Pipeline_Split_20261112_184956_Tô-Doanh-Quang-Khu.md
